# py-googletraffic: Getting Started

This notebook demonstrates how to use the `py-googletraffic` package to extract and analyze Google Maps traffic data.

## Prerequisites

1. **Google Maps API Key**: You need a Google Maps JavaScript API key. Get one at: https://developers.google.com/maps/get-started
2. **ChromeDriver**: Selenium requires ChromeDriver. Install it:
   - macOS: `brew install chromedriver`
   - Ubuntu: `sudo apt-get install chromium-chromedriver`
   - Or download from: https://chromedriver.chromium.org/

## 1. Setup and Installation

In [ ]:
# Install the package (if not already installed)
# !pip install -e .

In [ ]:
# Import required libraries
import googletraffic as gt
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
import warnings
warnings.filterwarnings('ignore')

# For visualization
%matplotlib inline

In [ ]:
# Set your Google Maps API key
GOOGLE_API_KEY = "YOUR_GOOGLE_MAPS_API_KEY_HERE"

# Note: Never commit your API key to version control!
# Consider using environment variables:
# import os
# GOOGLE_API_KEY = os.getenv('GOOGLE_MAPS_API_KEY')

## 2. Basic Example: Traffic Around a Point

Let's create a traffic raster centered on Times Square, New York City.

In [ ]:
# Define location (Times Square, NYC)
location = (40.7580, -73.9855)  # (latitude, longitude)

# Create traffic raster
traffic_raster = gt.make_raster(
    location=location,
    height=1000,
    width=1000,
    zoom=14,  # City-level detail
    google_key=GOOGLE_API_KEY,
    wait_time=3  # Wait 3 seconds for traffic layer to load
)

print(f"Raster shape: {traffic_raster.shape}")
print(f"Unique traffic levels: {np.unique(traffic_raster)}")

In [ ]:
# Visualize the traffic raster
plt.figure(figsize=(12, 10))

# Define colors matching Google Maps
colors = ['white', 'green', 'orange', 'red', 'darkred']
from matplotlib.colors import ListedColormap
cmap = ListedColormap(colors)

plt.imshow(traffic_raster, cmap=cmap, vmin=0, vmax=4)
plt.colorbar(label='Traffic Level', ticks=[0, 1, 2, 3, 4])
plt.title('Traffic Conditions Around Times Square, NYC', fontsize=14, fontweight='bold')
plt.xlabel('Pixel X')
plt.ylabel('Pixel Y')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='white', edgecolor='black', label='No data (0)'),
    Patch(facecolor='green', label='No traffic (1)'),
    Patch(facecolor='orange', label='Medium traffic (2)'),
    Patch(facecolor='red', label='High traffic (3)'),
    Patch(facecolor='darkred', label='Heavy traffic (4)')
]
plt.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

## 3. Save as GeoTIFF

Save the traffic raster as a georeferenced GeoTIFF file for use in GIS software.

In [ ]:
# Create and save traffic raster as GeoTIFF
output_file = gt.make_raster(
    location=(40.7580, -73.9855),
    height=1000,
    width=1000,
    zoom=14,
    google_key=GOOGLE_API_KEY,
    output_path='nyc_traffic.tif'
)

print(f"GeoTIFF saved to: {output_file}")

In [ ]:
# Read and display the GeoTIFF with geographic context
with rasterio.open('nyc_traffic.tif') as src:
    print("Raster metadata:")
    print(f"  - CRS: {src.crs}")
    print(f"  - Bounds: {src.bounds}")
    print(f"  - Transform: {src.transform}")
    print(f"  - Width x Height: {src.width} x {src.height}")
    
    # Plot with coordinates
    fig, ax = plt.subplots(figsize=(12, 10))
    show(src, ax=ax, cmap=cmap, vmin=0, vmax=4)
    ax.set_title('Traffic Raster with Geographic Coordinates')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.tight_layout()
    plt.show()

## 4. Traffic Analysis

Perform basic statistical analysis on the traffic data.

In [ ]:
# Calculate traffic statistics
traffic_pixels = traffic_raster[traffic_raster > 0]  # Exclude no-data pixels

if len(traffic_pixels) > 0:
    print("Traffic Statistics:")
    print(f"  - Total traffic pixels: {len(traffic_pixels):,}")
    print(f"  - Mean traffic level: {traffic_pixels.mean():.2f}")
    print(f"  - Median traffic level: {np.median(traffic_pixels):.2f}")
    print("\nTraffic distribution:")
    
    for level in range(1, 5):
        count = np.sum(traffic_raster == level)
        percentage = (count / len(traffic_pixels)) * 100
        level_names = ['', 'No traffic', 'Medium', 'High', 'Heavy']
        print(f"  - Level {level} ({level_names[level]}): {count:,} pixels ({percentage:.1f}%)")

In [ ]:
# Visualize traffic distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
traffic_counts = [np.sum(traffic_raster == i) for i in range(1, 5)]
ax1.bar(['No traffic', 'Medium', 'High', 'Heavy'], traffic_counts, 
        color=['green', 'orange', 'red', 'darkred'])
ax1.set_ylabel('Number of Pixels')
ax1.set_title('Traffic Level Distribution')
ax1.grid(axis='y', alpha=0.3)

# Pie chart
ax2.pie(traffic_counts, labels=['No traffic', 'Medium', 'High', 'Heavy'],
        colors=['green', 'orange', 'red', 'darkred'], autopct='%1.1f%%')
ax2.set_title('Traffic Level Proportions')

plt.tight_layout()
plt.show()

## 5. Bounding Box Example

Create a traffic raster for a custom bounding box.

In [ ]:
# Define bounding box for Lower Manhattan
# Format: (west, south, east, north)
bbox = (-74.02, 40.70, -73.97, 40.73)

print("Creating traffic raster for bounding box...")
print("This may take a moment and could require multiple API calls.")

traffic_bbox = gt.make_raster_from_bbox(
    bbox=bbox,
    zoom=14,
    google_key=GOOGLE_API_KEY,
    output_path='manhattan_traffic.tif'
)

print(f"Saved to: {traffic_bbox}")

## 6. Working with Polygons (Optional)

If you have a shapefile or GeoJSON, you can create traffic rasters from polygon boundaries.

In [ ]:
# Example with a custom polygon
import geopandas as gpd
from shapely.geometry import box

# Create a simple polygon (rectangle around Central Park)
polygon = box(-73.98, 40.76, -73.95, 40.80)
gdf = gpd.GeoDataFrame([1], geometry=[polygon], crs='EPSG:4326')

print("Creating traffic raster for polygon...")
traffic_poly = gt.make_raster_from_polygon(
    polygon=gdf,
    zoom=14,
    google_key=GOOGLE_API_KEY,
    output_path='central_park_traffic.tif'
)

print(f"Saved to: {traffic_poly}")

## 7. Time Series Analysis

Capture traffic at different times to analyze temporal patterns.

In [ ]:
# Capture traffic at different times (run this cell multiple times throughout the day)
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
output_path = f'nyc_traffic_{timestamp}.tif'

gt.make_raster(
    location=(40.7580, -73.9855),
    height=1000,
    width=1000,
    zoom=14,
    google_key=GOOGLE_API_KEY,
    output_path=output_path
)

print(f"Captured traffic at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Saved to: {output_path}")

## 8. Tips and Best Practices

### Zoom Levels
- **10-12**: Regional view (counties, large metro areas)
- **13-15**: City-level view (neighborhoods, districts)
- **16-18**: Street-level detail (individual roads)

### API Costs
- Each API call costs $0.007 (1,000 calls = $7)
- Google provides $200/month credit (~28,571 free calls)
- Large areas require multiple API calls

### Performance
- Increase `wait_time` if traffic layer doesn't load completely
- Use `headless=False` for debugging browser issues
- Adjust `max_pixels` to balance between API calls and image quality

### Data Quality
- Traffic data reflects real-time conditions
- Some areas may have limited traffic data coverage
- Capture during peak hours for more interesting results

## Next Steps

- Integrate with other geospatial data (demographics, points of interest)
- Build time-series models of traffic patterns
- Create interactive maps with folium
- Analyze traffic impacts on air quality or commute times